Generates data for the Van Der Pol System

In [ ]:
# DEPENDENCIES & GLOBAL VARIABLES
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from pathlib import Path

# random number generator, can set a seed for consistency
rng = np.random.default_rng()

# equation stuff
EQUATION = 'complete'
MU = 10 # mu value for van der pol equation
T_SPAN = [0,20] # time range to evaluate (was 20)
DT = 0.05 # time step (was 0.05)
print(f"Will generate {40000/(T_SPAN[1]/DT)} time series of length {T_SPAN[1]/DT}.")

# initial condition stuff
REJECT = 'none' # reject I.C.s inside/outside/none ellipse defined above

# ellipse of rejection
if REJECT == 'outside:': A, B = 0.5, 5 # ellipse inscribing the limit cycle curve
else: A, B = 2, 15 # ellipse circumscribing the limit cycle curve
# rectangle of choice, tuned so that 50%+-20% of choices are inside the limit cycle
# with mu=10, attractor can be approximated as $6.5\left(x+2\right)\sin\left(0.7\left(x+2\right)\right)-x-2$
C, D = 2.5, 16 

# saving stuff
FILE_PATH = Path('training_data/uhist-extraoutside.hdf5')
OVERWRITE = False
if FILE_PATH.exists() and not OVERWRITE: raise FileExistsError

In [ ]:
# HELPER FUNCTIONS

def vanderpol(t, u, mu):
    """Replicates Van Der Pol Equation"""
    # ut is time derivative of u
    ut = np.zeros_like(u)
    ut[0] = u[1]
    ut[1] = mu*(1-u[0]**2)*u[1] - u[0]
    return ut

def vanderpol_damped(t, u, mu):
    """Replicates Van Der Pol Equation without +muy term"""
    ut = np.zeros_like(u)
    ut[0] = u[1]
    ut[1] = mu*(-u[0]**2)*u[1] - u[0]
    return ut

def init_generator_ellipse(a, b):
    """Chooses a random point in the ellipse defined implicitly as x2/a2 + y2/b2 = 1"""
    # choose a random r and theta
    r = np.sqrt(rng.random()) # square rooted so that probability is still uniform
    theta = 2*np.pi*rng.random()

    # calulate and return the corresponding cartesian coords
    x = a*r*np.cos(theta)
    y = b*r*np.sin(theta)
    return np.array([x,y], dtype=np.float64)

def init_generator_rejection(a,b, c,d, reject):
    """Chooses a random point inside the rectangle with width 2c and height 2d
    May or may not reject points based on their location inside or outside the ellipse based on the value of reject 
    """
    while True:
        # generate random point inside the rectangle
        x = rng.uniform(-c, c)
        y = rng.uniform(-d, d)
        point = np.array([x,y], dtype=np.float64)

        # evaluate whether the point is inside the ellipse
        inside = ((x**2 / a**2) + (y**2 / b**2)) <= 1

        # evaluate whether the point should be rejected or whether we can move on with our lives
        if reject == 'none': break
        elif reject == 'inside' and not inside: break
        elif reject == 'outside' and inside: break

    return point

def init_generator_ellipse_rejectinisde(a, b, num):
    """Generates evenly space initial conditions along three concentric ellispses, and rejects if they're inside the curve:
    $6.5\\left(x+2\\right)\\sin\\left(0.7\\left(x+2\\right)\\right)-x-2$ or its 180 degree rotated counterpart"""

    f = lambda q: 6.5 * (q+2) * np.sin(0.7 * (q+2)) - q - 2

    initial_conditions = []
    thetas = np.linspace(0, 2*np.pi, num, endpoint=False)
    for theta in thetas:
        x = a*np.cos(theta)
        y = b*np.sin(theta)

        if not (y > -f(-x) and y < f(x)):
            initial_conditions.append([x, y])

    initial_conditions = np.array(initial_conditions)

    return initial_conditions
                

In [ ]:
# ACTUAL CALCULATIONS & PLOTTING

ts_eval = np.arange(T_SPAN[0], T_SPAN[1], DT)
plt.figure(figsize=(12,6))

# this stuff is only needed if you're doing the concentric ellipses
aa = A*np.array([0.75, 1.2, 1.6, 2.0])
bb = B*np.array([0.75, 1.2, 1.6, 2.0])
initial_conditions = np.concatenate((init_generator_ellipse_rejectinisde(aa[0], bb[0], 10), init_generator_ellipse_rejectinisde(aa[1], bb[1], 10), init_generator_ellipse_rejectinisde(aa[2], bb[2], 12), init_generator_ellipse_rejectinisde(aa[3], bb[3], 14)))
# plot ellipses of initial conditions (if you're doing that)
for a, b in zip(aa, bb):
    ellipse = mpatches.Ellipse((0,0), 2*a, 2*b, fill=False, edgecolor='lightgray', linewidth=2, linestyle=':')
    plt.gca().add_patch(ellipse)

#[init_generator_rejection(A,B, C,D, REJECT) for _ in range(int(40000/(T_SPAN[1]/DT)))] # ICS are random points inside an ellipse
func = vanderpol if EQUATION == 'complete' else (vanderpol_damped if EQUATION == 'damped' else None)

u = np.zeros((1,2)) # just need something to concatonate to later

# loop through all the initial conditions so we have all the data
for i, initial_condition in enumerate(initial_conditions):

    # use scipy's solve ivp to integrate system forward in time
    solution = solve_ivp(
        fun=func,
        t_span=T_SPAN,
        y0=initial_condition,
        args=(MU,),
        t_eval=ts_eval,    
    )

    # plot phase portrait for each initial condition
    plt.plot(solution.y[0], solution.y[1], alpha=0.6)

    # add phase portrait data to record
    u = np.concatenate((u, np.array([solution.y[0], solution.y[1]]).T), axis=0)

u = np.delete(u, (0), axis=0) # delete the first dummy row

plt.xlabel("$u_0$")
plt.ylabel("$u_1$")
plt.title("Van Der Pol Oscillator")
plt.show()

In [ ]:
# SAVE DATA TO H5 FILE

#df = pd.DataFrame(u)
#df.to_hdf(FILE_PATH.as_posix(), key='df', complevel=9)

In [ ]:
# VISUALIZE INITIAL CONDITIONS (NOT NECESSARY)

#initial_conditions = [init_generator(A,B) for _ in range(500)]
initial_conditions2 = np.array(initial_conditions)

plt.scatter(initial_conditions2[:, 0], initial_conditions2[:, 1], color='blue', alpha=0.5, s=5)
plt.xlabel("u[0]")
#plt.axis([-C,C,-D,D])
plt.ylabel("u[1] = d/dt u[0]")
plt.title("Van Der Pol Oscillator")
plt.grid(True)
plt.show()

In [ ]:
# SAVING AND PLOTTING GROUND STATE FIGURES

SAVE = True
OVERWRITE = False
EQUATION = 'damped'
OBSERVABLE_DATA = 'all'

load_file_path = 'training_data/uhist'
load_file_path += f'-{EQUATION}' if EQUATION != 'complete' else f""
load_file_path += f'-{OBSERVABLE_DATA}'
load_file_path += f'.hdf5'
load_file_path = Path(load_file_path)

if SAVE:
    save_file_path = 'phase_portraits/ground_state'
    save_file_path += f'-{EQUATION}' if EQUATION != 'complete' else f""
    save_file_path += f'-{OBSERVABLE_DATA}'
    save_file_path += f'.png'
    save_file_path = Path(save_file_path)
    if not OVERWRITE and save_file_path.exists(): 
        raise FileExistsError(f"{save_file_path} already exists. Change OVERWRITE to true if you would like to overwrite this file")

series_length_map = {
    'complete': 400,
    'damped': 400
}

# load data
data = pd.read_hdf(load_file_path, key='df').to_numpy()

# plot setup
plt.figure(figsize=(12, 6))

row_num = 0

while row_num < 40000:
    u = data[row_num:(row_num + series_length_map[EQUATION]), :]
    plt.plot(u[:, 0], u[:, 1], alpha=0.6)
    row_num += series_length_map[EQUATION]

# plot labels
plt.suptitle(
    f"Ground State: {EQUATION} eqn | {OBSERVABLE_DATA} I.C.s", 
    fontsize=14, fontweight='bold'
    )
plt.title('Phase portraits for several initial conditions', fontsize=10)
plt.xlabel('$u_0$', fontsize=12)
plt.ylabel('$u_1$', fontsize=12)
plt.tight_layout()

if SAVE: 
    plt.savefig(save_file_path, dpi=150)
    print("Plot saved successfully")
plt.show()

In [ ]:
# PLAIN 'OL SLOPE FIELD OF PHASE PLANE

xmax = 2.5
ymax = 16

# Apply arcsine function to bunch points near 0, then scale to your domain
x_grid = np.linspace(-xmax, xmax, 20)
y_grid = np.linspace(-ymax, ymax, 20)
X, Y = np.meshgrid(x_grid, y_grid)

# Van Der Pol Equations
dx = Y
dy = MU * (1 - X**2) * Y - X

# create figure
plt.figure(figsize=(8, 6))

plt.quiver(
    X, Y, dx, dy, 
    color="purple",
    pivot='middle'
)

plt.xlabel('$u_0$')
plt.ylabel('$u_1$')
plt.title("Phase Plane Vector Field")
plt.grid(True)
plt.show()

In [ ]:
# SCALED SLOPE FIELD OF PHASE PLANE

xmax = 20
ymax = 40

# Standard linear values between -1 and 1
evenly_x = np.linspace(-1, 1, 20)
evenly_y = np.linspace(-1, 1, 20)

# Apply arcsine function to bunch points near 0, then scale to your domain
x_grid = xmax * np.arcsin(evenly_x * 0.8)
y_grid = ymax * np.arcsin(evenly_y * 0.8)
X, Y = np.meshgrid(x_grid, y_grid)

# Van Der Pol Equations
dx = Y
dy = MU * (1 - X**2) * Y - X

# scaling wrt axes and normalization of vectors
dxu = dx / xmax
dyu = dy / ymax
norm = np.hypot(dxu, dyu)
norm[norm == 0] = 1.0  # Avoid zero division
norm = np.pow(norm, 14/16)
dxu = dxu / norm
dyu = dyu / norm

# create figure
plt.figure(figsize=(8, 6))

plt.quiver(
    X, Y, dxu, dyu, 
    color="purple",
    pivot='middle'
)

plt.xlabel('$u_0$')
plt.ylabel('$u_1$')
plt.title("Phase Plane Vector Field")
plt.grid(True)
plt.show()